<p></p>

In [14]:
import pandas as pd
print("pandas 导入成功！")

pandas 导入成功！


In [15]:
daily = pd.read_csv('daily_sales.csv')
print(daily.head())

  sale_date  order_count  total_revenue
0       NaN            1           0.00
1   1/10/11           38       14984.40
2   1/11/11           73       59430.32
3   1/12/11           55       16431.69
4   1/13/11           47       15080.21


In [16]:
daily = daily.dropna(subset=['sale_date'])
print(daily.head())

  sale_date  order_count  total_revenue
1   1/10/11           38       14984.40
2   1/11/11           73       59430.32
3   1/12/11           55       16431.69
4   1/13/11           47       15080.21
5   1/14/11           47       39332.38


In [17]:
daily['sale_date'] = pd.to_datetime(daily['sale_date'], format='%m/%d/%y')
daily = daily.sort_values('sale_date')
print(daily.head())

    sale_date  order_count  total_revenue
77 2010-12-01          127       46051.26
87 2010-12-02          160       45775.43
93 2010-12-03           64       22598.46
95 2010-12-05           94       31380.60
97 2010-12-06          111       30465.08


In [20]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(daily['sale_date'], daily['total_revenue'], marker='o', linestyle='-')
plt.title('Daily Total Revenue')
plt.xlabel('Date')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

<Figure size 864x432 with 1 Axes>

In [24]:
rfm = pd.read_csv('user_rfm.csv')
print(rfm.head())

   customer_id  last_purchase  frequency  monetary
0        12346  1/18/11 10:01          1  77183.60
1        12347    8/2/11 8:48          7   4310.00
2        12348  9/25/11 13:13          4   1797.24
3        12349  11/21/11 9:51          1   1757.55
4        12350   2/2/11 16:01          1    334.40


In [25]:
rfm['last_purchase'] = pd.to_datetime(rfm['last_purchase'], format='%m/%d/%y %H:%M', errors='coerce')
# 检查是否有缺失值
print(f"转换失败的行数：{rfm['last_purchase'].isna().sum()}")
# 查看最大日期
print(rfm['last_purchase'].max())

转换失败的行数：0
2011-12-09 12:16:00


In [29]:
# 计算 recency（天数差）
rfm['recency'] = (rfm['last_purchase'].max() - rfm['last_purchase']).dt.days

# 查看前几行确认
print(rfm[['customer_id', 'last_purchase', 'recency', 'frequency', 'monetary']].head())

   customer_id       last_purchase  recency  frequency  monetary
0        12346 2011-01-18 10:01:00      325          1  77183.60
1        12347 2011-08-02 08:48:00      129          7   4310.00
2        12348 2011-09-25 13:13:00       74          4   1797.24
3        12349 2011-11-21 09:51:00       18          1   1757.55
4        12350 2011-02-02 16:01:00      309          1    334.40


In [30]:
# 稳健的 RFM 打分函数（使用 rank 方法）
def rfm_score_r(x):
    # recency: 天数越少越好，升序分档（小值高分）
    return pd.qcut(x.rank(method='first'), 3, labels=[3,2,1])

def rfm_score_fm(x):
    # frequency/monetary: 值越大越好，降序分档（大值高分）
    return pd.qcut(x.rank(method='first', ascending=False), 3, labels=[3,2,1])

rfm['R'] = rfm_score_r(rfm['recency'])
rfm['F'] = rfm_score_fm(rfm['frequency'])
rfm['M'] = rfm_score_fm(rfm['monetary'])

# 组合 RFM 分数（字符串形式，例如 '333'）
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

# 查看前几行
print(rfm[['customer_id', 'recency', 'frequency', 'monetary', 'RFM_Score']].head())

   customer_id  recency  frequency  monetary RFM_Score
0        12346      325          1  77183.60       123
1        12347      129          7   4310.00       233
2        12348       74          4   1797.24       333
3        12349       18          1   1757.55       323
4        12350      309          1    334.40       121


In [31]:
# 筛选出 R、F、M 都是 3 的用户（最高价值）
high_value = rfm[(rfm['R'] == 3) & (rfm['F'] == 3) & (rfm['M'] == 3)]
print(f"高价值用户数量：{len(high_value)}")
print(high_value[['customer_id', 'monetary']].head())

# 计算高价值用户贡献的销售额占比
total_monetary = rfm['monetary'].sum()
high_value_monetary = high_value['monetary'].sum()
print(f"高价值用户贡献占比：{high_value_monetary / total_monetary * 100:.2f}%")

高价值用户数量：367
    customer_id  monetary
2         12348   1797.24
5         12352   2506.04
15        12362   5226.23
17        12364   1313.10
29        12380   2724.81
高价值用户贡献占比：23.89%


In [32]:
rfm.to_csv('rfm_with_scores.csv', index=False)
print("RFM 分层结果已保存")

RFM 分层结果已保存


In [33]:
high_value = rfm[(rfm['R'] == 3) & (rfm['F'] == 3) & (rfm['M'] == 3)]
print(f"高价值用户数量：{len(high_value)}")

total_monetary = rfm['monetary'].sum()
high_value_monetary = high_value['monetary'].sum()
print(f"高价值用户贡献占比：{high_value_monetary / total_monetary * 100:.2f}%")

高价值用户数量：367
高价值用户贡献占比：23.89%


In [34]:
rfm.to_csv('rfm_with_scores.csv', index=False)
print("RFM 分层结果已保存")

RFM 分层结果已保存
